<a href="https://colab.research.google.com/github/palakbhatt1/mistral-7b-retail-banking-chatbot-qlora/blob/main/mistral_7b_qlora_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# @title
!pip install -U transformers
!pip install -U datasets
!pip install -U accelerate
!pip install -U peft
!pip install -U trl==0.12.2
!pip install -U bitsandbytes
!pip install -U wandb
!pip install -U rouge-score nltk
print('All libraries installed!')


  Using cached transformers-5.5.0-py3-none-any.whl.metadata (32 kB)
Using cached transformers-5.5.0-py3-none-any.whl (10.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.4/637.4 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 73.5 MB/s eta 0:00:00
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.4.2
    Uninstalling hf-xet-1.4.2:
      Successfully uninstalled hf-xet-1.4.2
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.20.3
    Uninstalling tokenizers-0.20.3:
      Successfully uninstalled tokenizers-0.20.3
  Attempting uninstall: transformers
    Found existing installation: transformers 4.46.3
    Uninstalling transformers-4.46

In [ ]:
!pip install -q -U rouge-score evaluate nltk
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import login
import wandb

os.environ['HF_TOKEN']=userdata.get('HF_TOKEN')

login(token=os.environ['HF_TOKEN'])

print('Logged in to HuggingFace successfully!')

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in to HuggingFace successfully!


In [ ]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
)
from peft import (
    LoraConfig,
    PeftModel,
    prepare_model_for_kbit_training,
    get_peft_model,
)
import os, torch, wandb
from datasets import load_dataset, Dataset
from trl import SFTTrainer, setup_chat_format
import pandas as pd
import nltk
from nltk.translate.bleu_score import corpus_bleu, sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
import numpy as np
import warnings
warnings.filterwarnings('ignore')
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print('All imports successful!')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
  print(f'GPU: {torch.cuda.get_device_name(0)}')

All imports successful!
CUDA available: True
GPU: Tesla T4


In [ ]:
base_model = 'mistralai/Mistral-7B-v0.3'
new_model = 'Mistral-7B-v0.3-Retail-Banking-Bot'

LEARNING_RATE = 2e-4
BATCH_SIZE = 1
GRAD_ACCUM = 4
EPOCHS = 2
MAX_SEQ_LEN = 512
DATASET_SIZE = 1000
TEST_SPLIT = 0.1

print(f'Base Model: {base_model}')
print(f'Fine-tuned Model Name: {new_model}')
print(f'Training examples: {int(DATASET_SIZE * (1 - TEST_SPLIT))}')
print(f'Fine-tuned Model Name: {int(DATASET_SIZE * TEST_SPLIT)}')


Base Model: mistralai/Mistral-7B-v0.3
Fine-tuned Model Name: Mistral-7B-v0.3-Retail-Banking-Bot
Training examples: 900
Fine-tuned Model Name: 100


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(base_model)
model, tokenizer = setup_chat_format(model, tokenizer)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [ ]:

raw_ds = load_dataset("bitext/Bitext-retail-banking-llm-chatbot-training-dataset", split=f"train[:{DATASET_SIZE}]")
def preprocess_function(row):

  messages = [
      {"role":"user", "content": row["instruction"]},
      {"role": "assistant", "content": row["response"]}
  ]
  row["text"] = tokenizer.apply_chat_template(messages, tokenize=False)
  return row
dataset = raw_ds.map(preprocess_function)
dataset = dataset.train_test_split(test_size=TEST_SPLIT)

In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    peft_config=peft_config,
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    tokenizer=tokenizer,
    args=TrainingArguments(
        output_dir=new_model,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        num_train_epochs=EPOCHS,
        fp16=True,
        logging_steps=10,
        eval_strategy="steps",
    ),
)

trainer.train()

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss
10,0.906500,0.717946
20,0.609100,0.658460
30,0.623100,0.627920
40,0.619800,0.601969
50,0.550700,0.604633
60,0.561900,0.608695
70,0.569100,0.602926
80,0.560800,0.591191
90,0.526400,0.574486
100,0.531400,0.588425


TrainOutput(global_step=450, training_loss=0.49561609903971354, metrics={'train_runtime': 3210.2599, 'train_samples_per_second': 0.561, 'train_steps_per_second': 0.14, 'total_flos': 1.8966505815097344e+16, 'train_loss': 0.49561609903971354, 'epoch': 2.0})

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

save_path = "/content/drive/MyDrive/mistral_7b_qlora_finetuning"

trainer.model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


('/content/drive/MyDrive/mistral_7b_qlora_finetuning/tokenizer_config.json',
 '/content/drive/MyDrive/mistral_7b_qlora_finetuning/special_tokens_map.json',
 '/content/drive/MyDrive/mistral_7b_qlora_finetuning/tokenizer.json')

In [ ]:
scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

def generate_response(user_query):
    prompt = tokenizer.apply_chat_template([{"role": "user", "content": user_query}], tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=100)
    return tokenizer.decode(outputs[0], skip_special_tokens=True).split("assistant")[-1].strip()

test_query = "I haven't received my refund yet, what should I do?"
bot_response = generate_response(test_query)

print(f"\n--- Comparison Output ---\nUser: {test_query}\nBot: {bot_response}")




--- Comparison Output ---
User: I haven't received my refund yet, what should I do?
Bot: I'm here to assist you with your refund issue. I'm sorry for any inconvenience caused. Here's what you need to do:

1. Check the status of your refund on our website or mobile app. You can find it under the "Account" or "Profile" section.
2. If the status is still "Pending" or "In Progress," please be patient. Refunds can take a few days to process.
3. If the


eval/loss,█▆▅▄▄▄▄▄▄▃▃▄▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
eval/runtime,█▂▁▃▃▄▃▂▂▁▂▃▃▂▄▂▂▃▄▃▃▃▃▃▃▃▃▃▂▄▁▃▁▂▃▃▃▃▄▃
eval/samples_per_second,▁▇█▆▅▅▆▇██▇▆▅▇▅▇▇▆▅▆▆▆▆▆▇▅▅▆▇▅█▆█▇▆▆▆▆▅▆
eval/steps_per_second,▁▇█▆▆▆▆▇██▇▆▆▇▆▇▇▆▆▆▆▆▆▆▆▆▆▆▇▆█▆█▇▆▆▆▆▅▆
train/epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇█████
train/global_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇█████
train/grad_norm,█▆▇█▄▅▅▄▃▂▅▅▄▂▄▂▂▃▄▃▂▃▂▂▂▂▃▃▃▁▂▂▂▂▂▂▂▂▁▂
train/learning_rate,████▇▇▇▇▇▆▆▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▁▁▁
train/loss,█▄▄▄▃▃▃▃▃▂▃▃▃▃▂▃▃▃▃▃▂▂▂▁▁▁▂▂▁▁▁▁▁▁▁▁▁▁▁▁
eval/loss,0.5009
eval/runtime,34.4692


In [ ]:
import evaluate
from rouge_score import rouge_scorer

rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")

samples = dataset["test"].select(range(10))
predictions = []
references = [ex["response"] for ex in samples]

print("--- GENETATING COMPARISON OUTPUTS ---")

for i, example in enumerate(samples):
    query = example["instruction"]

    prompt = tokenizer.apply_chat_template([{"role": "user", "content": query}], tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=0.7)
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True).split("assistant")[-1].strip()

    predictions.append(generated_text)

    if i < 3:
        print(f"\n[Test Case {i+1}]")
        print(f"Customer Query: {query}")
        print(f"Expected Banking Response: {references[i]}")
        print(f"Model's Generated Response: {generated_text}")
        print("-" * 30)

rouge_results = rouge.compute(predictions=predictions, references=references)
bleu_results = bleu.compute(predictions=predictions, references=references)

print("\n--- FINAL COMPARISON METRICS ---")
print(f"ROUGE-1 (Content Match): {rouge_results['rouge1']:.4f}")
print(f"ROUGE-L (Sequence Match): {rouge_results['rougeL']:.4f}")
print(f"BLEU Score (Fluency): {bleu_results['bleu']:.4f}")

--- GENETATING COMPARISON OUTPUTS ---

[Test Case 1]
Customer Query: id like to activate an amex help me
Expected Banking Response: Without a doubt! I'm here to assist you with activating your American Express card. Activating your card is quick and easy. Here are the steps to get it done:

1. Visit the American Express website or open the {{Bank App}}.
2. Go to the "Activate Card" section. You will find this option under your account or in the main menu.
3. Enter the required information, such as your card number, security code, and any other details as prompted.
4. Review the terms and conditions, if provided, and accept them.
5. Once you've entered all the necessary details, submit the information.
6. Your card should now be activated, and you can start using it for your purchases.

If you encounter any issues or need further assistance during the activation process, feel free to reach out to our customer support team at {{Customer Support Phone Number}}. They'll be more than happy 

In [ ]:
print("--- CALCULATING TASK-SPECIFIC METRICS ---")

def calculate_task_accuracy(predictions):

    template_indicators = ["{{Credit Card}}", "{{Account Number}}", "I'm here to assist", "simple process"]

    match_count = 0
    for pred in predictions:
        if any(indicator in pred for indicator in template_indicators):
            match_count += 1

    accuracy = (match_count / len(predictions)) * 100
    return accuracy


task_acc = calculate_task_accuracy(predictions)

# 2. Human Evaluation Scorecard (Input your manual ratings here)
human_scores = {
    "Fluency": [5, 5, 4, 5, 5],        # Score 1-5 for each example
    "Banking Tone": [5, 5, 5, 4, 5],   # Score 1-5 for each example
    "Relevance": [5, 5, 5, 5, 5]       # Score 1-5 for each example
}

avg_human_eval = {k: sum(v)/len(v) for k, v in human_scores.items()}

print(f"\n--- EXTENDED METRICS FOR REPORT ---")
print(f"Task-Specific Accuracy (Template Match): {task_acc}%")
print(f"Human Evaluation - Average Fluency: {avg_human_eval['Fluency']}/5")
print(f"Human Evaluation - Average Banking Tone: {avg_human_eval['Banking Tone']}/5")
print(f"Human Evaluation - Average Relevance: {avg_human_eval['Relevance']}/5")

--- CALCULATING TASK-SPECIFIC METRICS ---

--- EXTENDED METRICS FOR REPORT ---
Task-Specific Accuracy (Template Match): 80.0%
Human Evaluation - Average Fluency: 4.8/5
Human Evaluation - Average Banking Tone: 4.8/5
Human Evaluation - Average Relevance: 5.0/5


In [ ]:
import shutil

src = "/content/Mistral-7B-v0.3-Retail-Banking-Bot"
dst = "/content/drive/MyDrive/mistral_7b_qlora_finetuning"

shutil.copytree(src, dst, dirs_exist_ok=True)

'/content/drive/MyDrive/mistral_7b_qlora_finetuning'

In [ ]:
pip install -U bitsandbytes>=0.46.1

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel, PeftConfig

adapter_path = "/content/drive/MyDrive/mistral_7b_qlora_finetuning/checkpoint-450"

# ✅ 4-bit config — fits Mistral 7B in ~6GB GPU RAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True  # saves extra memory
)

config = PeftConfig.from_pretrained(adapter_path)
base_model_name = config.base_model_name_or_path

tokenizer = AutoTokenizer.from_pretrained(adapter_path)

# ✅ Load in 4-bit — no more RAM crash
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

# ✅ Fix vocab size mismatch
base_model.resize_token_embeddings(len(tokenizer))

model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()
print("✅ Model loaded successfully!")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


✅ Model loaded successfully!


In [ ]:
def generate_response(user_query):

    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": user_query}],
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )


    return tokenizer.decode(outputs[0], skip_special_tokens=True).split("assistant")[-1].strip()


queries = [
    "i have received a new card and i wanna activate it what should i do",
    "help me to activate a visa"
]

for q in queries:
    print("Q:", q)
    print("A:", generate_response(q))
    print("-" * 50)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Q: i have received a new card and i wanna activate it what should i do


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


A: I'm here to assist you with activating your new card. Activating your card is a simple process. Here's what you need to do:

1. Locate the activation sticker on your new card. It is usually found on the front or back of the card.
2. Dial the activation phone number provided on the sticker or the activation instructions that came with your card.
3. Follow the automated prompts and enter the required information, such as
--------------------------------------------------
Q: help me to activate a visa
A: I'm here to assist you with activating your {{Credit Card}}. Activating your card is a simple process. Here's what you need to do:

1. Call the activation phone number provided on the sticker attached to your {{Credit Card}} or the activation number mentioned in the activation instructions.
2. Follow the prompts and enter the required information, such as your card number, expiration date, and security code.
3. Once you'
--------------------------------------------------


In [1]:
!git clone https://github.com/palakbhatt1/mistral-7b-retail-banking-chatbot-qlora.git

Cloning into 'mistral-7b-retail-banking-chatbot-qlora'...
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 9 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (9/9), 25.20 KiB | 5.04 MiB/s, done.
